#ARCHIVE AND PURGE PROCESS

In [0]:
from pyspark.sql.functions import *
from delta.tables import *
from pyspark.sql import Row
from datetime import datetime


In [0]:
# DEFINE STORAGE PATHS
active_path = "abfss://active-data@strgarchiveproject01.dfs.core.windows.net/"
archive_path = "abfss://archive-data@strgarchiveproject01.dfs.core.windows.net/"
logs_path = "abfss://logs@strgarchiveproject01.dfs.core.windows.net/"
silver_path = active_path + "silver_ecommerce"
archive_table_path = archive_path + "archived_silver"

In [0]:
# LOAD ACTIVE SILVER TABLE
silver_table = spark.read.format("delta") \
    .load(silver_path)

silver_table.select("InvoiceDate").show(10, False)

+-------------------+
|InvoiceDate        |
+-------------------+
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
|2011-04-07 12:16:00|
+-------------------+
only showing top 10 rows



In [0]:
# RETENTION POLICY
retention_expr = "current_date() - INTERVAL 5 YEARS"
# IDENTIFY OLD RECORDS
archivable_records = silver_table.filter(
    col("InvoiceDate") < expr(retention_expr)
)
archive_count = archivable_records.count()
print(f"Records identified for archive: {archive_count}")
display(archivable_records)

Records identified for archive: 541909


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,ingestion_date,record_status
549251,84596F,SMALL MARSHMALLOWS PINK BOWL,2,2011-04-07T12:16:00Z,0.42,14449,United Kingdom,2026-05-22,ACTIVE
549251,21975,PACK OF 60 DINOSAUR CAKE CASES,1,2011-04-07T12:16:00Z,0.55,14449,United Kingdom,2026-05-22,ACTIVE
549251,22417,PACK OF 60 SPACEBOY CAKE CASES,1,2011-04-07T12:16:00Z,0.55,14449,United Kingdom,2026-05-22,ACTIVE
549251,22326,ROUND SNACK BOXES SET OF4 WOODLAND,1,2011-04-07T12:16:00Z,2.95,14449,United Kingdom,2026-05-22,ACTIVE
549251,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2011-04-07T12:16:00Z,2.95,14449,United Kingdom,2026-05-22,ACTIVE
549251,85093,CANDY SPOT EGG WARMER HARE,10,2011-04-07T12:16:00Z,0.39,14449,United Kingdom,2026-05-22,ACTIVE
549251,85094,CANDY SPOT EGG WARMER RABBIT,5,2011-04-07T12:16:00Z,0.19,14449,United Kingdom,2026-05-22,ACTIVE
549251,85132B,CHARLIE AND LOLA TABLE TINS,2,2011-04-07T12:16:00Z,1.95,14449,United Kingdom,2026-05-22,ACTIVE
549251,85132C,CHARLIE AND LOLA FIGURES TINS,2,2011-04-07T12:16:00Z,1.95,14449,United Kingdom,2026-05-22,ACTIVE
549251,22138,BAKING SET 9 PIECE RETROSPOT,1,2011-04-07T12:16:00Z,4.95,14449,United Kingdom,2026-05-22,ACTIVE


In [0]:
# ARCHIVE RECORDS
if archive_count > 0:

    archivable_records.write.format("delta") \
        .mode("append") \
        .save(archive_table_path)

    print("Records archived successfully")
else:
    print("No records found for archival")

Records archived successfully


In [0]:
# DELETE ARCHIVED RECORDS
delta_table = DeltaTable.forPath(
    spark,
    silver_path
)
delta_table.delete(
    f"InvoiceDate < {retention_expr}"
)
print("Archived records deleted from active table")

Archived records deleted from active table


In [0]:
# DISABLE RETENTION CHECK
spark.conf.set(
    "spark.databricks.delta.retentionDurationCheck.enabled",
    "false"
)

In [0]:
# VACUUM PURGE
delta_table.vacuum(0)
print("VACUUM completed successfully")

VACUUM completed successfully


In [0]:
# VALIDATION
active_count = spark.read.format("delta") \
    .load(silver_path) \
    .count()
archived_count = spark.read.format("delta") \
    .load(archive_table_path) \
    .count()
print("===================================")
print("VALIDATION REPORT")
print("===================================")
print(f"Active Records   : {active_count}")
print(f"Archived Records : {archived_count}")

VALIDATION REPORT
Active Records   : 0
Archived Records : 541909


In [0]:
#DELTA HISTORY
print("===================================")
print("DELTA HISTORY")
print("===================================")

spark.sql(f"""
DESCRIBE HISTORY delta.`{silver_path}`
""").show(truncate=False)


DELTA HISTORY
+-------+-------------------+---------------+-------------------------------------------------------+------------+----------------------------------------------------------------------------------------------------+----+------------------+--------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation   |operationParameters                                                                                 |job |notebook          |clusterId           |readVersion|isolationLevel   |isBlindAppe

In [0]:
# CREATE LIFECYCLE LOGS
log_data = [Row(
    execution_time=str(datetime.now()),
    archived_records=archived_count,
    active_records=active_count,
    status="SUCCESS"
)]
log_df = spark.createDataFrame(log_data)
log_df.write.format("delta") \
    .mode("append") \
    .save(logs_path + "lifecycle_logs")
print("Lifecycle logs created successfully")
print("===================================")
print("ARCHIVE AND PURGE PIPELINE COMPLETED")
print("===================================")

Lifecycle logs created successfully
ARCHIVE AND PURGE PIPELINE COMPLETED
